# Getting Started with AstroML

AstroML is a dynamic graph machine learning framework for the Stellar blockchain.
This notebook walks you through the core workflow:

1. Setting up the environment
2. Ingesting and normalizing ledger data
3. Building a transaction graph
4. Extracting node features
5. Running a baseline GCN model

## 1. Installation

```bash
git clone https://github.com/Traqora/astroml.git
cd astroml
python -m venv venv && source venv/bin/activate
pip install -r requirements.txt
```

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import networkx as nx
print('AstroML environment ready')

## 2. Ledger Data Ingestion

AstroML ingests Stellar ledger data and normalizes it into a standard format.
For this tutorial we use synthetic data that mirrors the real schema.

In [ ]:
from astroml.ingestion.normalizer import normalize_transaction

# Example raw Stellar payment operation
raw_tx = {
    'id': 'tx_001',
    'ledger_sequence': 1000000,
    'created_at': '2024-01-15T10:30:00Z',
    'source_account': 'GABC1234',
    'operations': [{
        'type': 'payment',
        'from': 'GABC1234',
        'to': 'GXYZ5678',
        'asset_code': 'XLM',
        'amount': '100.0',
    }]
}

normalized = normalize_transaction(raw_tx)
print('Normalized transaction:')
for k, v in normalized.items():
    print(f'  {k}: {v}')

## 3. Building a Transaction Graph

Transactions become directed edges between account nodes.

In [ ]:
from astroml.features.transaction_graph import TransactionGraph

# Build a graph from a list of normalized transactions
transactions = [
    {'from': 'GABC', 'to': 'GXYZ', 'amount': 100.0, 'asset': 'XLM', 'timestamp': 1000},
    {'from': 'GXYZ', 'to': 'GDEF', 'amount': 50.0,  'asset': 'XLM', 'timestamp': 1001},
    {'from': 'GABC', 'to': 'GDEF', 'amount': 200.0, 'asset': 'USDC', 'timestamp': 1002},
    {'from': 'GDEF', 'to': 'GABC', 'amount': 30.0,  'asset': 'XLM', 'timestamp': 1003},
]

tg = TransactionGraph()
for tx in transactions:
    tg.add_transaction(**tx)

G = tg.to_networkx()
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Nodes: {list(G.nodes())}')
print(f'Edges: {list(G.edges(data=True))}')

## 4. Node Feature Extraction

AstroML computes rich per-account features from the transaction graph.

In [ ]:
from astroml.features.node_features import compute_node_features

features = compute_node_features(G)
print('Node features (first account):')
first_node = list(features.keys())[0]
for feat, val in features[first_node].items():
    print(f'  {feat}: {val:.4f}')

## 5. Training a Baseline GCN

Use the built-in GCN model for node classification.

In [ ]:
# Train using the CLI-equivalent Python API
# Equivalent to: python -m astroml.training.train_gcn

from astroml.training.train_gcn import train_gcn

# Uses the Cora dataset by default for demonstration
result = train_gcn(epochs=10, hidden_dim=64, lr=0.01)
print(f'Training complete. Final accuracy: {result["accuracy"]:.4f}')

## Next Steps

- **`02_fraud_detection.ipynb`** — Detect fraud patterns using GNNs and Deep SVDD
- **`03_transaction_graph_analysis.ipynb`** — Temporal graph analysis and anomaly detection
- See `docs/` for full API reference and architecture overview